# Vault Headline Scanner — token-free (RSS)

Pulls headlines from the majors **for free** (RSS + Google News RSS), tags each by which **vault thread**
it hits, dedups, sorts newest-first, and prints a grouped digest. No API keys, no tokens, no cost — you
run it in Colab. Read it, or paste the digest back to Claude to ingest.

**Three sources:**
1. **Google News RSS search** (the backbone) — free, and it *aggregates the paywalled majors*
   (Reuters, Bloomberg, WSJ) that killed their own free feeds. One query → everyone's headline on it.
2. **Direct free RSS** — CNN, Fox, ABC, CNBC, MarketWatch, WSJ, Guardian.
3. **Per-ticker Yahoo Finance RSS** — for your watch names.

---
### Honest limits (read once)
- **X/Twitter is NOT token-free** — the free API is dead and Nitter mirrors are unreliable. Stub left
  below; don't count on it. (To watch @zerohedge etc. you'd need the paid X API.)
- **Bloomberg/Reuters direct feeds are paywalled/dead** — you get their *headlines* via Google News
  aggregation, but the linked article may be behind a paywall. Fine for a headline scan.
- RSS gives headline + short summary, **not full articles**. This surfaces & filters; the *analysis* is
  you reading it or pasting the digest back (that keeps it token-free).
- Google News RSS is free but unofficial — stable for years, not guaranteed forever. A dead feed is
  skipped, not fatal.


In [ ]:
import sys, subprocess
try:
    import feedparser
except Exception:
    subprocess.run([sys.executable,'-m','pip','install','-q','feedparser'])
    import feedparser
import urllib.parse, time, re
from datetime import datetime, timezone


## CONFIG — vault threads (keywords), search topics, feeds, tickers


In [ ]:
# Each headline gets tagged by which THREAD its text matches. Edit keywords to retune.
THREADS = {
  'WAR/OIL/HORMUZ': ['iran','hormuz','bandar abbas','houthi','red sea','bab el-mandeb','strait','tanker',
      'basra','opec','brent','crude','oil','refinery','tehran','israel','gulf','kuwait','qeshm','ahvaz'],
  'MEMORY/SEMIS':   ['micron','memory','dram','hbm','nand','sk hynix','samsung','tsmc','semiconductor',
      'chip','nvidia','broadcom','avgo','sox','wafer'],
  'AI-CAPEX/FRAGILITY': ['ai capex','hyperscaler','data center','datacenter','capex','depreciation',
      'cover ratio','buyback','openai','anthropic','ai bubble','rubber band','deleveraging'],
  'FED/INFLATION/RATES': ['federal reserve',' fed ','powell','warsh','cpi','ppi','inflation','rate cut',
      'rate hike','fomc','treasury yield','30-year','10-year','pce','jobless'],
  'KOREA/LEVERAGE': ['kospi','korea','margin call','levered etf','carry trade','deleverage','liquidation'],
  'POWER/GRID':     ['power grid','electricity','nuclear','smr','transformer','pjm','grid','fusion','uranium'],
  'GOLD/DEBASEMENT':['gold','debasement','devalue','dollar','stablecoin','bitcoin','remonetiz','crypto'],
  'EARNINGS':       ['earnings','guidance','revenue','profit','beats','misses','forecast','netflix','nflx'],
}

# Google News search topics (the backbone — catches Reuters/Bloomberg/WSJ via aggregation)
GNEWS_QUERIES = [
  'Iran Hormuz oil strait tanker', 'Micron memory DRAM chip shortage', 'AI capex hyperscaler data center',
  'Federal Reserve inflation rate cut', 'Korea KOSPI margin call leverage', 'power grid datacenter electricity',
  'gold dollar debasement', 'stock market selloff earnings', 'Brent crude oil price', 'Nvidia Broadcom semiconductor',
]

# Direct free RSS feeds
DIRECT_FEEDS = {
  'CNBC-top':'https://search.cnbc.com/rs/search/combinedcms/view.xml?partnerId=wrss01&id=100003114',
  'CNBC-mkts':'https://search.cnbc.com/rs/search/combinedcms/view.xml?partnerId=wrss01&id=15839069',
  'MarketWatch':'http://feeds.marketwatch.com/marketwatch/topstories/',
  'CNN-money':'http://rss.cnn.com/rss/money_latest.rss',
  'Fox-biz':'https://moxie.foxbusiness.com/google-publisher/markets.xml',
  'Fox-news':'https://moxie.foxnews.com/google-publisher/latest.xml',
  'ABC-money':'https://abcnews.go.com/abcnews/moneyheadlines',
  'ABC-intl':'https://abcnews.go.com/abcnews/internationalheadlines',
  'WSJ-mkts':'https://feeds.a.dj.com/rss/RSSMarketsMain.xml',
  'WSJ-world':'https://feeds.a.dj.com/rss/RSSWorldNews.xml',
  'Guardian-biz':'https://www.theguardian.com/business/rss',
}

TICKERS = ['MU','NVDA','AVGO','TSM','GOOGL','MSFT','AAPL','META','AMZN','TSLA','NFLX','LLY','NOW','SPY']
HOURS_BACK = 24   # only show headlines newer than this

# X/Twitter stub — NOT token-free; left disabled on purpose.
X_ENABLED = False


## Fetch + tag


In [ ]:
def gnews_url(q): return 'https://news.google.com/rss/search?q='+urllib.parse.quote(q)+'&hl=en-US&gl=US&ceid=US:en'
def yahoo_url(t): return 'https://feeds.finance.yahoo.com/rss/2.0/headline?s='+t+'&region=US&lang=en-US'

def entry_time(e):
    for k in ('published_parsed','updated_parsed'):
        if getattr(e, k, None): return datetime.fromtimestamp(time.mktime(getattr(e,k)), tz=timezone.utc)
    return None

def tag_threads(text):
    t = text.lower()
    return [name for name,kws in THREADS.items() if any(k in t for k in kws)]

def pull(url, source, want_all=False):
    out=[]
    try:
        f = feedparser.parse(url)
        for e in f.entries[:40]:
            title = getattr(e,'title',''); summ = re.sub('<[^<]+?>','',getattr(e,'summary',''))[:200]
            dt = entry_time(e)
            tags = tag_threads(title+' '+summ)
            if not want_all and not tags: continue          # keep only vault-relevant for broad feeds
            out.append(dict(title=title.strip(), source=source, link=getattr(e,'link',''), dt=dt, tags=tags))
    except Exception as ex:
        print('  skip', source, ex)
    return out

rows=[]
print('Google News topics...');   [rows.extend(pull(gnews_url(q),'GoogleNews')) for q in GNEWS_QUERIES]
print('Direct feeds...');         [rows.extend(pull(u,s)) for s,u in DIRECT_FEEDS.items()]
print('Per-ticker (Yahoo)...');   [rows.extend(pull(yahoo_url(t),'YF:'+t, want_all=True)) for t in TICKERS]
print('Pulled', len(rows), 'tagged rows')


## Dedup, filter by recency, print grouped digest


In [ ]:
# dedup by normalized title
seen=set(); uniq=[]
for r in sorted(rows, key=lambda r:(r['dt'] or datetime(1970,1,1,tzinfo=timezone.utc)), reverse=True):
    key = re.sub('[^a-z0-9]','', r['title'].lower())[:60]
    if key in seen or not key: continue
    seen.add(key); uniq.append(r)

now = datetime.now(timezone.utc)
def fresh(r): return (r['dt'] is None) or ((now-r['dt']).total_seconds() <= HOURS_BACK*3600)
uniq = [r for r in uniq if fresh(r)]

def age(r):
    if not r['dt']: return '  ?  '
    m=(now-r['dt']).total_seconds()/60
    return f'{int(m)}m' if m<90 else f'{int(m/60)}h'

print('='*70); print(f'  VAULT HEADLINE SCAN  ({now.astimezone().strftime("%Y-%m-%d %H:%M %Z")}, last {HOURS_BACK}h)'); print('='*70)
for thread in list(THREADS.keys()):
    hits=[r for r in uniq if thread in r['tags']]
    if not hits: continue
    print(f'\n### {thread}  ({len(hits)})')
    for r in hits[:12]:
        print(f'  [{age(r):>4}] {r["title"][:96]}  — {r["source"]}')

untagged=[r for r in uniq if not r['tags']]   # only YF ticker rows land here (kept via want_all)
if untagged:
    print(f'\n### TICKER-SPECIFIC (untagged, {len(untagged)})')
    for r in untagged[:15]: print(f'  [{age(r):>4}] {r["title"][:96]}  — {r["source"]}')
print('\n' + '='*70)
print('Paste this digest back to Claude to ingest, or open a link to read.')


### Notes
- **Retune** by editing `THREADS` keywords / `GNEWS_QUERIES` / `HOURS_BACK` in CONFIG.
- To add a source: drop its RSS URL into `DIRECT_FEEDS`. To add a ticker: add to `TICKERS`.
- Links are printed with each headline in the row dict (`r['link']`) — add `r["link"]` to the print if
  you want clickable URLs; left off by default to keep the digest scannable on a phone.
- **X/Twitter**: `X_ENABLED=False` and intentionally not wired — no free path exists. If you ever get a
  paid X API key, that's a separate build.
